# 🍽️ Predictive Restaurant Recommender
### Location-Aware Vendor Recommendation using Haversine Distance & Order History

---

**Author:** Sireesha Ragipati  
**Dataset:** Soulpage IT Solutions — Food Delivery Platform  
**Algorithm:** Haversine Distance-Based Ranking + Order History Scoring  
**Tech Stack:** Python · Pandas · NumPy · Scikit-learn

---

## 📌 Project Overview

This project builds a **Predictive Restaurant Recommendation System** for a food delivery platform. Given a customer's GPS location, the system recommends the **Top-N most relevant vendors (restaurants)** by combining:

1. **Proximity** — Haversine distance between customer and vendor
2. **Historical signals** — vendor ratings, order frequency, favorites, discounts

### Dataset Summary

| Table | Rows | Description |
|-------|------|-------------|
| `train_customers` | 34,674 | Customer profiles |
| `train_locations` | 59,503 | Customer GPS coordinates |
| `train_orders` | 135,303 | Historical order transactions |
| `vendors` | 100 | Restaurant metadata |
| `test_customers` | 9,768 | Test customer profiles |
| `test_locations` | 16,720 | Test customer GPS coordinates |

---
## 📦 Step 0: Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load all tables
train_customers = pd.read_csv("Train_soulpage/train_customers.csv")
train_locations = pd.read_csv("Train_soulpage/train_locations.csv")
train_orders    = pd.read_csv("Train_soulpage/orders.csv", low_memory=False)
vendors         = pd.read_csv("Train_soulpage/vendors.csv")

print("✅ All training files loaded successfully!")
print(f"   train_customers : {train_customers.shape}")
print(f"   train_locations : {train_locations.shape}")
print(f"   train_orders    : {train_orders.shape}")
print(f"   vendors         : {vendors.shape}")

✅ All training files loaded successfully!
   train_customers : (34674, 8)
   train_locations : (59503, 5)
   train_orders    : (135303, 26)
   vendors         : (100, 59)


---
## 🧹 Step 1: Clean `train_customers`

### Column-by-Column Analysis

| Column | Action | Reason |
|--------|--------|--------|
| `customer_id` | ✅ Keep | Primary key for all joins |
| `gender` | ❌ Drop | 35% missing, highly imbalanced (~93% Male), weak signal |
| `dob` | ❌ Drop | 91% missing — imputing would create noise |
| `status` | ❌ Drop | 99.9% have status=1, zero variation |
| `verified` | ❌ Drop | Not related to restaurant preferences |
| `language` | ❌ Drop | Only 'EN' values, no variation |
| `created_at` | ❌ Drop | Account creation date doesn't predict food preference |
| `updated_at` | ❌ Drop | Last profile update — no predictive value |

In [2]:
print("📊 train_customers — Before Cleaning:")
print(f"   Shape: {train_customers.shape}")
print(f"   Missing values:\n{train_customers.isnull().sum()}")

📊 train_customers — Before Cleaning:
   Shape: (34674, 8)
   Missing values:
customer_id        0
gender         12154
dob            31628
status             0
verified           0
language       13575
created_at         0
updated_at         0
dtype: int64


In [3]:
# Clean gender column (has trailing spaces, mixed cases)
train_customers['gender'] = (
    train_customers['gender']
    .str.strip()
    .str.capitalize()
    .replace(['', '?????'], np.nan)
)

print("Gender value counts after cleaning:")
print(train_customers['gender'].value_counts(dropna=False))

Gender value counts after cleaning:
gender
Male      20738
NaN       12157
Female     1779
Name: count, dtype: int64


In [4]:
# Drop all non-essential columns
train_customers.drop(
    ['gender', 'dob', 'status', 'verified', 'language', 'created_at', 'updated_at'],
    axis=1, inplace=True
)

# Remove 151 duplicate customer_id entries
train_customers.drop_duplicates(subset='customer_id', inplace=True)

print(f"✅ train_customers cleaned: {train_customers.shape}")
print(f"   Unique customers: {train_customers['customer_id'].nunique()}")
train_customers.head(2)

✅ train_customers cleaned: (34523, 1)
   Unique customers: 34523


,customer_id
0,TCHWPBT
1,ZGFSYCZ


---
## 🗺️ Step 2: Clean `train_locations`

### Column-by-Column Analysis

| Column | Action | Reason |
|--------|--------|--------|
| `customer_id` | ✅ Keep | Links location to customer |
| `location_number` | ✅ Keep | Distinguishes home, work, other addresses |
| `location_type` | ❌ Drop | 46% missing, highly uneven (Home dominates) |
| `latitude` | ✅ Keep | Critical for proximity-based recommendations |
| `longitude` | ✅ Keep | Critical for proximity-based recommendations |

> **Key Insight:** Each customer can have multiple locations (home, work, etc.). The `customer_id + location_number` pair is always unique — no deduplication needed on this table.

In [5]:
print("📊 train_locations — Before Cleaning:")
print(f"   Shape: {train_locations.shape}")
print(f"   Unique customers: {train_locations['customer_id'].nunique()}")
print(f"   Missing values:\n{train_locations.isnull().sum()}")

📊 train_locations — Before Cleaning:
   Shape: (59503, 5)
   Unique customers: 35400
   Missing values:
customer_id            0
location_number        0
location_type      27209
latitude               6
longitude              6
dtype: int64


In [6]:
# Verify no duplicate (customer_id + location_number) pairs
dup_count = train_locations.duplicated(subset=['customer_id', 'location_number']).sum()
print(f"Duplicate (customer_id, location_number) pairs: {dup_count}")

# Drop only 6 rows where lat/lon is null
train_locations.dropna(subset=['latitude', 'longitude'], inplace=True)
train_locations.drop(['location_type'], axis=1, inplace=True)
train_locations.reset_index(drop=True, inplace=True)

print(f"\n✅ train_locations cleaned: {train_locations.shape}")
train_locations.head(2)

Duplicate (customer_id, location_number) pairs: 0

✅ train_locations cleaned: (59497, 4)


,customer_id,location_number,latitude,longitude
0,02SFNJH,0,1.682392,-78.789737
1,02SFNJH,1,1.679137,0.766823


---
## 🛒 Step 3: Clean `train_orders`

### Column Decisions

| Column | Action | Reason |
|--------|--------|--------|
| `customer_id`, `vendor_id` | ✅ Keep | Core linkage columns |
| `item_count` | ✅ Keep — fill median | Shows order size; 5% missing |
| `grand_total` | ✅ Keep | Order value — customer spending signal |
| `vendor_discount_amount` | ✅ Keep | Customers prefer discounts |
| `is_favorite` | ✅ Keep — encode 0/1 | Direct preference signal |
| `is_rated` | ✅ Keep — encode 0/1 | Engagement indicator |
| `vendor_rating` | ✅ Keep — fill 0 | Quality signal; add `has_vendor_rating` flag |
| `deliverydistance` | ✅ Keep | Distance affects customer choice |
| `LOCATION_NUMBER` | ✅ Keep | Needed for merging with locations |
| `promo_code` | ❌ Drop | 97% missing |
| All timestamp columns | ❌ Drop | 37–96% missing; sparse |
| `payment_mode`, `driver_rating` etc. | ❌ Drop | Not predictive for vendor choice |

In [7]:
print("📊 train_orders — Before Cleaning:")
print(f"   Shape: {train_orders.shape}")
print(f"   Missing values (top 10):")
print(train_orders.isnull().sum().sort_values(ascending=False).head(10))

📊 train_orders — Before Cleaning:
   Shape: (135303, 26)
   Missing values (top 10):
promo_code                        130998
delivery_time                     130180
delivery_date                      99759
vendor_rating                      90083
driver_accepted_time               88845
promo_code_discount_percentage     69423
preparationtime                    55560
picked_up_time                     51438
ready_for_pickup_time              51054
delivered_time                     49562
dtype: int64


In [8]:
# Remove 70 rows with null order_id, then deduplicate
train_orders = train_orders.dropna(subset=['order_id'])
train_orders = train_orders.drop_duplicates(subset=['order_id'], keep='first')

print(f"After deduplication: {train_orders.shape}")
print(f"Duplicate order_ids remaining: {train_orders['order_id'].duplicated().sum()}")

After deduplication: (135221, 26)
Duplicate order_ids remaining: 0


In [9]:
# Drop columns that are sparse, redundant, or non-predictive
cols_to_drop = [
    'order_id', 'payment_mode', 'promo_code', 'driver_rating',
    'promo_code_discount_percentage', 'created_at', 'delivery_date',
    'LOCATION_TYPE', 'CID X LOC_NUM X VENDOR', 'preparationtime',
    'delivery_time', 'order_accepted_time', 'driver_accepted_time',
    'ready_for_pickup_time', 'picked_up_time', 'delivered_time'
]
train_orders.drop(cols_to_drop, axis=1, inplace=True)

# --- Fill missing values ---
# item_count: 5% missing → fill with median
train_orders['item_count'] = train_orders['item_count'].fillna(train_orders['item_count'].median())

# is_favorite: NaN likely means 'not favorited'
train_orders['is_favorite'] = train_orders['is_favorite'].fillna('No')

# Encode Yes/No → 1/0
train_orders['is_favorite'] = train_orders['is_favorite'].map({'Yes': 1, 'No': 0})
train_orders['is_rated']    = train_orders['is_rated'].map({'Yes': 1, 'No': 0})

# vendor_rating: 66% missing → create flag + fill 0
train_orders['has_vendor_rating'] = train_orders['vendor_rating'].notna().astype(int)
train_orders['vendor_rating'].fillna(0, inplace=True)

print(f"✅ train_orders cleaned: {train_orders.shape}")
print(f"   Missing values remaining:\n{train_orders.isnull().sum()}")
train_orders.head(2)

✅ train_orders cleaned: (135221, 11)
   Missing values remaining:
customer_id               0
item_count                0
grand_total               0
vendor_discount_amount    0
is_favorite               0
is_rated                  0
vendor_rating             0
deliverydistance          0
vendor_id                 0
LOCATION_NUMBER           0
has_vendor_rating         0
dtype: int64


,customer_id,item_count,grand_total,vendor_discount_amount,is_favorite,is_rated,vendor_rating,deliverydistance,vendor_id,LOCATION_NUMBER,has_vendor_rating
0,KL09J9N,6.0,10.1,0.0,0,0,0.0,0.0,84,0,0
1,H5LGGFX,3.0,8.4,0.0,0,0,0.0,0.0,78,0,0


---
## 🏪 Step 4: Clean `vendors`

### Column Decisions

| Column | Action | Reason |
|--------|--------|--------|
| `id`, `latitude`, `longitude` | ✅ Keep | Vendor identity + location |
| `vendor_rating` | ✅ Keep | Quality signal |
| `prepration_time`, `delivery_charge` | ✅ Keep | Operational features customers care about |
| `serving_distance`, `is_open` | ✅ Keep | Feasibility indicators |
| `discount_percentage` | ✅ Keep | Preference signal |
| `vendor_tag_name` | ✅ Keep | Cuisine type — key for content-based filtering |
| `vendor_category_en` | ✅ Keep — encode | Restaurant vs Sweets & Bakes |
| All 28 day-of-week time columns | ❌ Drop | Already captured by `is_open` |
| `commission`, `is_haked_delivering` | ❌ Drop | Internal operational fields |
| `country_id`, `city_id`, `display_orders` | ❌ Drop | All same value — zero variation |

In [10]:
print("📊 vendors — Before Cleaning:")
print(f"   Shape: {vendors.shape}")
print(f"   Category distribution:")
print(vendors['vendor_category_en'].value_counts())

📊 vendors — Before Cleaning:
   Shape: (100, 59)
   Category distribution:
vendor_category_en
Restaurants       88
Sweets & Bakes    12
Name: count, dtype: int64


In [11]:
# Drop day-of-week schedule columns and other non-predictive columns
day_cols = []
for day in ['sunday','monday','tuesday','wednesday','thursday','friday','saturday']:
    for slot in ['from_time1','to_time1','from_time2','to_time2']:
        day_cols.append(f'{day}_{slot}')

other_drop = [
    'authentication_id', 'commission', 'is_haked_delivering', 'open_close_flags',
    'one_click_vendor', 'device_type', 'status', 'vendor_tag', 'updated_at',
    'vendor_category_id', 'OpeningTime', 'OpeningTime2', 'country_id', 'city_id',
    'display_orders', 'language', 'created_at', 'primary_tags'
]

vendors.drop(day_cols + other_drop, axis=1, inplace=True, errors='ignore')

# One-hot encode vendor category
vendors = pd.get_dummies(vendors, columns=['vendor_category_en'], drop_first=True)

# Drop 3 rows with no vendor_tag_name
vendors.dropna(subset=['vendor_tag_name'], inplace=True)

print(f"✅ vendors cleaned: {vendors.shape}")
print(f"   Columns: {list(vendors.columns)}")
vendors.head(2)

✅ vendors cleaned: (97, 13)
   Columns: ['id', 'latitude', 'longitude', 'delivery_charge', 'serving_distance', 'is_open', 'prepration_time', 'discount_percentage', 'verified', 'rank', 'vendor_rating', 'vendor_tag_name', 'vendor_category_en_Sweets & Bakes']


,id,latitude,longitude,delivery_charge,serving_distance,is_open,prepration_time,discount_percentage,verified,rank,vendor_rating,vendor_tag_name,vendor_category_en_Sweets & Bakes
0,4,-0.588596,0.754434,0.0,6,1,15,0,1,11,4.4,"Arabic,Breakfast,Burgers,Desserts,Free Deliver...",False
1,13,-0.471654,0.744470,0.7,5,1,14,0,1,11,4.7,"Breakfast,Cakes,Crepes,Italian,Pasta,Pizzas,Sa...",False


---
## 🔗 Step 5: Merge All Tables

```
train_orders  ←→  train_customers  (on customer_id)
     ↓
train_orders  ←→  train_locations  (on customer_id + LOCATION_NUMBER)
     ↓
train_orders  ←→  vendors          (on vendor_id)
```

In [12]:
# Start with orders as the base
merged_df = train_orders.copy()

# Merge customer info
merged_df = merged_df.merge(train_customers, on='customer_id', how='left')

# Merge customer location (match on customer + location number)
merged_df = merged_df.merge(
    train_locations,
    left_on=['customer_id', 'LOCATION_NUMBER'],
    right_on=['customer_id', 'location_number'],
    how='left'
)

# Merge vendor info
merged_df = merged_df.merge(
    vendors,
    left_on='vendor_id',
    right_on='id',
    how='left',
    suffixes=('', '_vendor')
)

print(f"✅ Merged dataset: {merged_df.shape}")
print(f"   Missing vendor coords: {merged_df['latitude_vendor'].isnull().sum()}")
print(f"   Missing customer coords: {merged_df['latitude'].isnull().sum()}")
merged_df.head(2)

✅ Merged dataset: (135221, 27)
   Missing vendor coords: 2181
   Missing customer coords: 5


,customer_id,item_count,grand_total,vendor_discount_amount,is_favorite,is_rated,vendor_rating,deliverydistance,vendor_id,LOCATION_NUMBER,has_vendor_rating,location_number,latitude,longitude,id,latitude_vendor,longitude_vendor,delivery_charge,serving_distance,is_open,prepration_time,discount_percentage,verified,rank,vendor_rating_vendor,vendor_tag_name,vendor_category_en_Sweets & Bakes
0,KL09J9N,6.0,10.1,0.0,0,0,0.0,0.0,84,0,0,0.0,-0.09065,-78.580196,84.0,-1.004923,0.078736,0.0,15.0,1.0,14.0,0.0,1.0,11.0,4.3,"Burgers,Fries,Kids meal,Shawarma",False
1,H5LGGFX,3.0,8.4,0.0,0,0,0.0,0.0,78,0,0,0.0,1.73395,-78.795830,78.0,-0.555404,0.196336,0.7,15.0,0.0,17.0,0.0,1.0,11.0,4.4,"Pizzas,Italian,Breakfast,Soups,Pasta,Salads,De...",False


---
## 📊 Step 6: Exploratory Data Analysis

Understanding patterns before building the model.

In [13]:
# Top 10 most ordered vendors
vendor_order_counts = merged_df['vendor_id'].value_counts().head(10)
print("🏆 Top 10 Most Ordered Vendors:")
print("-" * 35)
for vendor, count in vendor_order_counts.items():
    bar = '█' * (count // 200)
    print(f"  Vendor {vendor:<5} : {count:>5} orders  {bar}")

🏆 Top 10 Most Ordered Vendors:
-----------------------------------
  Vendor 113   :  7807 orders  ███████████████████████████████████████
  Vendor 105   :  5561 orders  ███████████████████████████
  Vendor 79    :  5112 orders  █████████████████████████
  Vendor 84    :  5001 orders  █████████████████████████
  Vendor 78    :  4643 orders  ███████████████████████
  Vendor 83    :  3684 orders  ██████████████████
  Vendor 386   :  3274 orders  ████████████████
  Vendor 86    :  2833 orders  ██████████████
  Vendor 846   :  2537 orders  ████████████
  Vendor 106   :  2263 orders  ███████████


In [14]:
# Delivery distance distribution
dist = merged_df['deliverydistance']
print("📦 Delivery Distance Stats (km):")
print("-" * 35)
print(f"  Mean   : {dist.mean():.2f} km")
print(f"  Median : {dist.median():.2f} km")
print(f"  Max    : {dist.max():.2f} km")
print(f"  Orders with 0 km distance: {(dist == 0).sum()} ({(dist == 0).mean()*100:.1f}%)")

📦 Delivery Distance Stats (km):
-----------------------------------
  Mean   : 4.10 km
  Median : 2.90 km
  Max    : 19.81 km
  Orders with 0 km distance: 55603 (41.1%)


In [15]:
# Vendor rating analysis
print("⭐ Vendor Rating Distribution (from orders):")
print("-" * 35)
rating_dist = merged_df[merged_df['has_vendor_rating'] == 1]['vendor_rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    bar = '▓' * (count // 500)
    print(f"  Rating {int(rating)} : {count:>5}  {bar}")

print(f"\n  % Orders rated : {(merged_df['has_vendor_rating'] == 1).mean()*100:.1f}%")
print(f"  % Favorited    : {(merged_df['is_favorite'] == 1).mean()*100:.1f}%")

⭐ Vendor Rating Distribution (from orders):
-----------------------------------
  Rating 0 : 25175  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  Rating 1 :  1029  ▓▓
  Rating 2 :   630  ▓
  Rating 3 :  1426  ▓▓
  Rating 4 :  2748  ▓▓▓▓▓
  Rating 5 : 14212  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  % Orders rated : 33.4%
  % Favorited    : 1.2%


In [16]:
# Vendor category breakdown
sweets_col = 'vendor_category_en_Sweets & Bakes'
if sweets_col in merged_df.columns:
    category_counts = merged_df[sweets_col].value_counts()
    print("🍽️ Orders by Vendor Category:")
    print("-" * 35)
    total = len(merged_df)
    for cat, cnt in category_counts.items():
        label = 'Sweets & Bakes' if cat else 'Restaurants'
        print(f"  {label:<20} : {cnt:>6} orders ({cnt/total*100:.1f}%)")

🍽️ Orders by Vendor Category:
-----------------------------------
  Restaurants          : 119980 orders (88.7%)
  Sweets & Bakes       :  13060 orders (9.7%)


---
## 🏗️ Step 7: Build Vendor Scoring System

Before predicting on test data, we build a **vendor score** per vendor based on historical order data. This score combines:

| Signal | Weight | Logic |
|--------|--------|-------|
| `vendor_rating` | High | Average rating per vendor |
| `is_favorite` | High | % of orders where customer marked as favorite |
| `is_rated` | Medium | % of orders that received a rating (engagement) |
| `vendor_discount_amount` | Medium | Average discount offered |
| `order_frequency` | High | Total number of times vendor was ordered |

# 
Build vendor popularity/quality score from historical data
vendor_stats = merged_df.groupby('vendor_id').agg(
    order_frequency       = ('vendor_id', 'count'),
    avg_rating            = ('vendor_rating', 'mean'),
    favorite_rate         = ('is_favorite', 'mean'),
    rating_engagement     = ('is_rated', 'mean'),
    avg_discount          = ('vendor_discount_amount', 'mean'),
    avg_items             = ('item_count', 'mean'),
    avg_spend             = ('grand_total', 'mean')
).reset_index()

# Normalize scores to 0–1 range
for col in ['order_frequency', 'avg_rating', 'favorite_rate', 'rating_engagement', 'avg_discount']:
    min_val = vendor_stats[col].min()
    max_val = vendor_stats[col].max()
    if max_val > min_val:
        vendor_stats[f'{col}_norm'] = (vendor_stats[col] - min_val) / (max_val - min_val)
    else:
        vendor_stats[f'{col}_norm'] = 0

# Composite vendor score
vendor_stats['vendor_score'] = (
    0.35 * vendor_stats['order_frequency_norm'] +
    0.30 * vendor_stats['avg_rating_norm'] +
    0.20 * vendor_stats['favorite_rate_norm'] +
    0.10 * vendor_stats['rating_engagement_norm'] +
    0.05 * vendor_stats['avg_discount_norm']
)

print("✅ Vendor scoring complete!")
print(f"   Vendors scored: {len(vendor_stats)}")
print("\n🏆 Top 10 Vendors by Composite Score:")
print("-" * 55)
top_vendors = vendor_stats.sort_values('vendor_score', ascending=False).head(10)
print(f"{'Vendor':<10} {'Orders':>8} {'Avg Rating':>12} {'Fav Rate':>10} {'Score':>8}")
print("-" * 55)
for _, row in top_vendors.iterrows():
    print(f"{int(row.vendor_id):<10} {int(row.order_frequency):>8} {row.avg_rating:>12.2f} {row.favorite_rate:>10.3f} {row.vendor_score:>8.4f}")

In [21]:
# Build vendor popularity/quality score from historical data
vendor_stats = merged_df.groupby('vendor_id').agg(
    order_frequency       = ('vendor_id', 'count'),
    avg_rating            = ('vendor_rating', 'mean'),
    favorite_rate         = ('is_favorite', 'mean'),
    rating_engagement     = ('is_rated', 'mean'),
    avg_discount          = ('vendor_discount_amount', 'mean'),
    avg_items             = ('item_count', 'mean'),
    avg_spend             = ('grand_total', 'mean')
).reset_index()

# Normalize scores to 0–1 range
for col in ['order_frequency', 'avg_rating', 'favorite_rate', 'rating_engagement', 'avg_discount']:
    min_val = vendor_stats[col].min()
    max_val = vendor_stats[col].max()
    if max_val > min_val:
        vendor_stats[f'{col}_norm'] = (vendor_stats[col] - min_val) / (max_val - min_val)
    else:
        vendor_stats[f'{col}_norm'] = 0

# Composite vendor score
vendor_stats['vendor_score'] = (
    0.35 * vendor_stats['order_frequency_norm'] +
    0.30 * vendor_stats['avg_rating_norm'] +
    0.20 * vendor_stats['favorite_rate_norm'] +
    0.10 * vendor_stats['rating_engagement_norm'] +
    0.05 * vendor_stats['avg_discount_norm']
)

print("✅ Vendor scoring complete!")
print(f"   Vendors scored: {len(vendor_stats)}")
print("\n🏆 Top 10 Vendors by Composite Score:")
print("-" * 55)
top_vendors = vendor_stats.sort_values('vendor_score', ascending=False).head(10)
print(f"{'Vendor':<10} {'Orders':>8} {'Avg Rating':>12} {'Fav Rate':>10} {'Score':>8}")
print("-" * 55)
for _, row in top_vendors.iterrows():
    print(f"{int(row.vendor_id):<10} {int(row.order_frequency):>8} {row.avg_rating:>12.2f} {row.favorite_rate:>10.3f} {row.vendor_score:>8.4f}")

✅ Vendor scoring complete!
   Vendors scored: 100

🏆 Top 10 Vendors by Composite Score:
-------------------------------------------------------
Vendor       Orders   Avg Rating   Fav Rate    Score
-------------------------------------------------------
79             5112         1.02      0.022   0.6283
113            7807         0.62      0.012   0.5824
849            1066         1.08      0.040   0.5555
573            1503         1.05      0.038   0.5450
386            3274         1.06      0.020   0.5441
855             831         1.08      0.035   0.5152
84             5001         0.78      0.015   0.5149
537            1255         1.33      0.015   0.5123
679            1125         1.16      0.022   0.4846
105            5561         0.63      0.011   0.4723


---
## 🧭 Step 8: Haversine Distance Function

The Haversine formula calculates the **great-circle distance** between two points on a sphere given their latitude/longitude:

$$d = 2R \cdot \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$$

Where $R$ = 6371 km (Earth's radius), $\phi$ = latitude, $\lambda$ = longitude.

> **Why Haversine over Euclidean?**  
> Euclidean distance is inaccurate for GPS coordinates because the Earth is a sphere, not a flat plane. Haversine accounts for Earth's curvature.

In [22]:
def haversine_single(lat1_rad, lon1_rad, lat2_arr, lon2_arr):
    """
    Vectorized Haversine distance from one point to many vendors.
    
    Parameters:
    -----------
    lat1_rad, lon1_rad : float
        Customer coordinates in radians
    lat2_arr, lon2_arr : np.array
        Vendor coordinates in radians
    
    Returns:
    --------
    np.array : Distances in kilometers
    """
    R = 6371  # Earth radius in km
    dlat = lat2_arr - lat1_rad
    dlon = lon2_arr - lon1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_arr) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))  # clip to avoid NaN from floating point
    return R * c

print("✅ Haversine distance function defined.")

# Quick test
# New York (40.7°N, 74.0°W) → London (51.5°N, 0.1°W) ≈ 5,570 km
ny_lat, ny_lon = np.radians(40.7), np.radians(-74.0)
lon_lat, lon_lon = np.radians(51.5), np.radians(-0.1)
test_dist = haversine_single(ny_lat, ny_lon, np.array([lon_lat]), np.array([lon_lon]))
print(f"\nSanity check — NY to London: {test_dist[0]:.0f} km (expected ≈ 5,570 km)")

✅ Haversine distance function defined.

Sanity check — NY to London: 5573 km (expected ≈ 5,570 km)


---
## 🧪 Step 9: Prepare Test Data

In [23]:
# Load test data
test_customers = pd.read_csv("Test_soulpage/test_customers.csv")
test_locations = pd.read_csv("Test_soulpage/test_locations.csv")

print(f"test_customers : {test_customers.shape}")
print(f"test_locations : {test_locations.shape}")

test_customers : (9768, 8)
test_locations : (16720, 5)


In [24]:
# Apply same cleaning steps as train_customers
test_customers.drop(
    ['gender', 'dob', 'status', 'verified', 'language', 'created_at', 'updated_at'],
    axis=1, inplace=True
)
test_customers.drop_duplicates(subset='customer_id', inplace=True)

# Apply same cleaning steps as train_locations
test_locations.dropna(subset=['latitude', 'longitude'], inplace=True)
test_locations.drop(['location_type'], axis=1, inplace=True)
test_locations.reset_index(drop=True, inplace=True)

# Merge test customers with their locations
test_data = pd.merge(test_customers, test_locations, on='customer_id', how='left')

# Drop one customer with invalid/missing location
test_data.dropna(subset=['latitude', 'longitude'], inplace=True)
test_data.reset_index(drop=True, inplace=True)

print(f"✅ Test data prepared: {test_data.shape}")
print(f"   Unique test customers: {test_data['customer_id'].nunique()}")
test_data.head(3)

✅ Test data prepared: (16312, 4)
   Unique test customers: 9752


,customer_id,location_number,latitude,longitude
0,ICE2DJP,0.0,-96.407538,-67.197291
1,ICE2DJP,1.0,0.038654,-78.595477
2,ICE2DJP,2.0,-95.106078,43.684151


---
## 🤖 Step 10: Generate Recommendations

### Recommendation Strategy

For each test customer location:
1. Compute Haversine distance to all unique vendors
2. Normalize distances to 0–1 (lower = closer = better)
3. Combine with vendor composite score: `final_score = 0.6 × proximity + 0.4 × vendor_score`
4. Return Top-N vendors ranked by final score

In [25]:
# Prepare training set for vendor coordinates
train_df = merged_df.dropna(subset=['latitude_vendor', 'longitude_vendor', 'latitude', 'longitude']).copy()

# Unique vendors with their coordinates + scores
vendor_coords = train_df[['vendor_id', 'latitude_vendor', 'longitude_vendor']].drop_duplicates('vendor_id')
vendor_coords = vendor_coords.merge(vendor_stats[['vendor_id', 'vendor_score']], on='vendor_id', how='left')
vendor_coords['vendor_score'] = vendor_coords['vendor_score'].fillna(0)

# Pre-compute radians for all vendors
vendor_lat_rad = np.radians(vendor_coords['latitude_vendor'].values)
vendor_lon_rad = np.radians(vendor_coords['longitude_vendor'].values)
vendor_ids     = vendor_coords['vendor_id'].values
vendor_scores  = vendor_coords['vendor_score'].values

TOP_N = 5

def get_recommendations(lat, lon, top_n=TOP_N):
    """
    Returns Top-N vendor recommendations for a given GPS coordinate.
    Combines proximity score (60%) + vendor quality score (40%).
    """
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    
    distances = haversine_single(lat_rad, lon_rad, vendor_lat_rad, vendor_lon_rad)
    
    # Normalize distance (closer = higher score)
    max_dist = distances.max()
    if max_dist > 0:
        proximity_score = 1 - (distances / max_dist)
    else:
        proximity_score = np.ones_like(distances)
    
    # Final score: proximity + vendor quality
    final_score = 0.6 * proximity_score + 0.4 * vendor_scores
    
    top_indices = np.argsort(final_score)[::-1][:top_n]
    
    return [
        {
            'vendor_id': int(vendor_ids[i]),
            'distance_km': round(distances[i], 2),
            'final_score': round(final_score[i], 4)
        }
        for i in top_indices
    ]

print(f"✅ Recommendation engine ready!")
print(f"   Total unique vendors available: {len(vendor_coords)}")

✅ Recommendation engine ready!
   Total unique vendors available: 97


---
## 🧪 Step 11: Demo — Recommendations for Sample Customers

In [26]:
# Demo: Show recommendations for 3 sample test customers
sample_customers = test_data.groupby('customer_id').first().reset_index().head(3)

for _, row in sample_customers.iterrows():
    cust_id = row['customer_id']
    lat, lon = row['latitude'], row['longitude']
    
    recs = get_recommendations(lat, lon, top_n=5)
    
    print(f"\n{'='*60}")
    print(f"👤 Customer: {cust_id}")
    print(f"📍 Location: ({lat:.4f}, {lon:.4f})")
    print(f"\n🍽️  Top 5 Recommended Restaurants:")
    print(f"{'Rank':<6} {'Vendor ID':<12} {'Distance (km)':>14} {'Score':>8}")
    print("-" * 45)
    for i, rec in enumerate(recs, 1):
        # Get vendor tags if available
        tag = vendors[vendors['id'] == rec['vendor_id']]['vendor_tag_name'].values
        tag_str = tag[0][:30] + '...' if len(tag) > 0 and len(str(tag[0])) > 30 else (tag[0] if len(tag) > 0 else 'N/A')
        print(f"  {i:<4} Vendor {rec['vendor_id']:<8} {rec['distance_km']:>10.2f} km  {rec['final_score']:>8.4f}")
        print(f"       Cuisine: {tag_str}")


👤 Customer: 000IPH5
📍 Location: (-0.3865, -78.5452)

🍽️  Top 5 Recommended Restaurants:
Rank   Vendor ID     Distance (km)    Score
---------------------------------------------
  1    Vendor 79          8793.05 km    0.2531
       Cuisine: Burgers,Desserts,Free Delivery...
  2    Vendor 113         8793.02 km    0.2348
       Cuisine: Arabic,Desserts,Free Delivery,...
  3    Vendor 849         8725.76 km    0.2286
       Cuisine: American,Breakfast,Burgers,Caf...
  4    Vendor 386         8742.03 km    0.2230
       Cuisine: Churros
  5    Vendor 573         8819.45 km    0.2180
       Cuisine: Burgers,Desserts,Free Delivery...

👤 Customer: 002U0H9
📍 Location: (-1.6549, -78.3953)

🍽️  Top 5 Recommended Restaurants:
Rank   Vendor ID     Distance (km)    Score
---------------------------------------------
  1    Vendor 79          8778.46 km    0.2533
       Cuisine: Burgers,Desserts,Free Delivery...
  2    Vendor 113         8778.43 km    0.2350
       Cuisine: Arabic,Desserts,Free De

---
## 📤 Step 12: Generate Full Submission File

In [27]:
import csv

submission_rows = []

for _, row in test_data.iterrows():
    cust_id  = row['customer_id']
    loc_num  = int(row['location_number'])
    lat, lon = row['latitude'], row['longitude']
    
    recs = get_recommendations(lat, lon, top_n=TOP_N)
    
    for rec in recs:
        submission_rows.append({
            'CID'    : cust_id,
            'LOC_NUM': loc_num,
            'VENDOR' : rec['vendor_id'],
            'target' : 0
        })

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('submission_final.csv', index=False)

print(f"✅ Submission file saved: submission_final.csv")
print(f"   Total rows: {len(submission_df)}")
print(f"   Unique customers: {submission_df['CID'].nunique()}")
print(f"   Unique vendors recommended: {submission_df['VENDOR'].nunique()}")
submission_df.head(10)

✅ Submission file saved: submission_final.csv
   Total rows: 81560
   Unique customers: 9752
   Unique vendors recommended: 16


,CID,LOC_NUM,VENDOR,target
0,ICE2DJP,0,79,0
1,ICE2DJP,0,849,0
2,ICE2DJP,0,113,0
3,ICE2DJP,0,386,0
4,ICE2DJP,0,84,0
5,ICE2DJP,1,79,0
6,ICE2DJP,1,113,0
7,ICE2DJP,1,849,0
8,ICE2DJP,1,386,0
9,ICE2DJP,1,573,0


---
## 📊 Step 13: Submission Analysis

In [28]:
# Which vendors got recommended most?
print("🏆 Most Frequently Recommended Vendors:")
print("-" * 40)
rec_counts = submission_df['VENDOR'].value_counts().head(10)
for vendor, count in rec_counts.items():
    # Get cuisine info
    tag = vendors[vendors['id'] == vendor]['vendor_tag_name'].values
    tag_str = tag[0][:35] if len(tag) > 0 else 'N/A'
    print(f"  Vendor {vendor:<5} : {count:>5} recommendations")
    print(f"             Cuisine: {tag_str}")

🏆 Most Frequently Recommended Vendors:
----------------------------------------
  Vendor 79    : 15189 recommendations
             Cuisine: Burgers,Desserts,Free Delivery,Past
  Vendor 113   : 14204 recommendations
             Cuisine: Arabic,Desserts,Free Delivery,India
  Vendor 386   : 12600 recommendations
             Cuisine: Churros
  Vendor 537   :  8481 recommendations
             Cuisine: American,Burgers,Desserts,Free Deli
  Vendor 849   :  7482 recommendations
             Cuisine: American,Breakfast,Burgers,Cafe,Des
  Vendor 573   :  7346 recommendations
             Cuisine: Burgers,Desserts,Free Delivery,Past
  Vendor 84    :  4268 recommendations
             Cuisine: Burgers,Fries,Kids meal,Shawarma
  Vendor 679   :  3226 recommendations
             Cuisine: Biryani,Desserts,Indian,Kebabs,Rice
  Vendor 295   :  1499 recommendations
             Cuisine: Coffee,Fresh Juices,Hot Chocolate,S
  Vendor 855   :  1442 recommendations
             Cuisine: American,Burgers,

In [29]:
# How many recommendations per customer?
recs_per_customer = submission_df.groupby('CID')['VENDOR'].count()
print("📋 Recommendations Per Customer:")
print("-" * 35)
print(f"  Min : {recs_per_customer.min()}")
print(f"  Max : {recs_per_customer.max()}")
print(f"  Mean: {recs_per_customer.mean():.1f}")
print(f"\n  Distribution:")
print(recs_per_customer.value_counts().sort_index())

📋 Recommendations Per Customer:
-----------------------------------
  Min : 5
  Max : 60
  Mean: 8.4

  Distribution:
VENDOR
5     6018
10    2228
15     796
20     378
25     168
30      94
35      44
40      15
45       7
50       2
55       1
60       1
Name: count, dtype: int64


---
## ✅ Step 14: Summary & Conclusions

### What We Built

| Component | Details |
|-----------|----------|
| **Data Cleaning** | 4 tables cleaned; dropped 40+ irrelevant columns with documented rationale |
| **Feature Engineering** | Vendor composite score from order history (frequency, ratings, favorites) |
| **Distance Model** | Haversine formula for GPS-accurate proximity calculation |
| **Recommendation Logic** | 60% proximity + 40% vendor quality hybrid scoring |
| **Output** | Top-5 vendor recommendations per customer-location pair |

### Key Findings

- **Location is the strongest signal** — customers overwhelmingly order from nearby vendors
- **Vendor ratings are sparse (66% missing)** but where present, they strongly predict repeat orders
- **Favorite marking is rare (~1.2%)** but is a high-quality positive signal when present
- **Same vendors recommended across similar locations** — shows geographic clustering in vendor placement

### Limitations & Next Steps

| Limitation | Potential Fix |
|------------|---------------|
| Location-only signals ignore food preferences | Add cuisine preference features from `vendor_tag_name` |
| No collaborative filtering | Use customer order history to find similar users |
| Some lat/lon values appear invalid (e.g., 205°) | Add coordinate validation + outlier filtering |
| Static scoring | Build time-aware model (order recency, time-of-day) |

---

### 🔗 Connect

- **LinkedIn:** [linkedin.com/in/sireesha-ragipati](https://www.linkedin.com/in/sireesha-ragipati-269a10244/)